In [ ]:
!pip install scikit-learn joblib

import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from joblib import dump


In [ ]:
!mkdir -p /content/_MusicProject
!unzip -q features_only.zip -d /content/_MusicProject
!ls -R /content/_MusicProject

/content/_MusicProject:
data

/content/_MusicProject/data:
features

/content/_MusicProject/data/features:
general  study

/content/_MusicProject/data/features/general:
'1st Contact - Duty Free.npz'
'99 Tales - Baby Out Of Jail.npz'
'A. Cooper - Aggression.npz'
'A. Cooper - Support.npz'
'Alaclair Ensemble - LICORNES.npz'
'Allister Thompson - The Old Miner.npz'
'Andrea - Work the Middle (Real Remix).npz'
'An Eagle In Your Mind - Cave of the Darling.npz'
'Austin Leonard Jones - Florida.npz'
'batchbug-snowflake(chosic.com).npz'
'Beat Mekanik - Harley Davidson.npz'
'BJ Block & Dawn Pemberton - Turn It Around.npz'
'Breuss Arrizabalaga Quintet - Tsurugi.npz'
'BrokeMC - broke Pimpin.npz'
'Bryan Mathys - Hard Miles.npz'
'Carlos Javed - Tiger - Roar of Destiny.npz'
'Carroll - Goat.npz'
'Clan Destined - Never All Ways (clean).npz'
'Cletus Got Shot - Charlie Jones.npz'
'Combo - The Lake.npz'
"Cullah - I'll Never Understand.npz"
'DARKNESS(chosic.com).npz'
'Dasnaty - I gat you.npz'
'Dead Times - Sl

In [ ]:
BASE_DIR = Path("/content/_MusicProject")
FEATURE_DIR = BASE_DIR / "data" / "features"

def load_all_features():
    X = []
    y = []
    paths = []

    for split_name, label in [("study", 1), ("general", 0)]:
        folder = FEATURE_DIR / split_name
        npz_files = sorted(folder.glob("*.npz"))
        if not npz_files:
            print(f"[WARN] No .npz files found in {folder}")
            continue

        for fpath in npz_files:
            data = np.load(fpath, allow_pickle=True)
            X.append(data["features"])
            y.append(label)
            paths.append(str(data["path"]))

    X = np.stack(X)
    y = np.array(y, dtype=int)
    paths = np.array(paths)

    print(f"[INFO] Loaded {X.shape[0]} tracks, {X.shape[1]} features each.")
    print(f"[INFO] Class balance: study={np.sum(y==1)}, general={np.sum(y==0)}")

    return X, y, paths

X, y, paths = load_all_features()


[INFO] Loaded 236 tracks, 44 features each.
[INFO] Class balance: study=122, general=114


In [ ]:
X_train, X_temp, y_train, y_temp, paths_train, paths_temp = train_test_split(
    X,
    y,
    paths,
    test_size=0.30,
    stratify=y,
    random_state=42,
)

X_val, X_test, y_val, y_test, paths_val, paths_test = train_test_split(
    X_temp,
    y_temp,
    paths_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=43,
)

print("[INFO] Train size:", X_train.shape[0])
print("[INFO] Val size:", X_val.shape[0])
print("[INFO] Test size:", X_test.shape[0])

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

logreg = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="lbfgs",
)

logreg.fit(X_train_s, y_train)

y_val_pred = logreg.predict(X_val_s)
y_val_proba = logreg.predict_proba(X_val_s)[:, 1]

val_acc = accuracy_score(y_val, y_val_pred)
val_auc = roc_auc_score(y_val, y_val_proba)

print("\n[VAL] Accuracy:", round(val_acc, 3))
print("[VAL] ROC-AUC:", round(val_auc, 3))
print("\n[VAL] Classification report:")
print(classification_report(y_val, y_val_pred, digits=3))

[INFO] Train size: 165
[INFO] Val size: 35
[INFO] Test size: 36

[VAL] Accuracy: 0.886
[VAL] ROC-AUC: 0.938

[VAL] Classification report:
              precision    recall  f1-score   support

           0      0.842     0.941     0.889        17
           1      0.938     0.833     0.882        18

    accuracy                          0.886        35
   macro avg      0.890     0.887     0.886        35
weighted avg      0.891     0.886     0.886        35



In [ ]:
y_test_pred = logreg.predict(X_test_s)
y_test_proba = logreg.predict_proba(X_test_s)[:, 1]

test_acc = accuracy_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_proba)

print("\n[TEST] Accuracy:", round(test_acc, 3))
print("[TEST] ROC-AUC:", round(test_auc, 3))
print("\n[TEST] Classification report:")
print(classification_report(y_test, y_test_pred, digits=3))

MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

dump(scaler, MODEL_DIR / "focus_scaler_logreg.joblib")
dump(logreg, MODEL_DIR / "focus_logreg.joblib")

print("\n[INFO] Saved:")
print(MODEL_DIR / "focus_scaler_logreg.joblib")
print(MODEL_DIR / "focus_logreg.joblib")



[TEST] Accuracy: 0.917
[TEST] ROC-AUC: 0.969

[TEST] Classification report:
              precision    recall  f1-score   support

           0      0.938     0.882     0.909        17
           1      0.900     0.947     0.923        19

    accuracy                          0.917        36
   macro avg      0.919     0.915     0.916        36
weighted avg      0.918     0.917     0.916        36


[INFO] Saved:
/content/_MusicProject/models/focus_scaler_logreg.joblib
/content/_MusicProject/models/focus_logreg.joblib


In [ ]:
for i in range(min(10, len(y_test_proba))):
    proba = float(y_test_proba[i])
    print(paths_test[i], "-> focusability score:", round(proba, 3), "label:", y_test[i])

data/processed/general/Lila Protocoll - Yorkshire Tea.wav -> focusability score: 0.004 label: 0
data/processed/general/The Defibulators - Holy Roller.wav -> focusability score: 0.067 label: 0
data/processed/study/tokyo-music-walker-sunset-drive-chosic.com_.wav -> focusability score: 0.433 label: 1
data/processed/general/The Pop Winds - Owl Eyes.wav -> focusability score: 0.003 label: 0
data/processed/study/Illusions(chosic.com).wav -> focusability score: 0.957 label: 1
data/processed/general/The Sharp Things - Homeless.wav -> focusability score: 0.53 label: 0
data/processed/study/Night-Owls(chosic.com).wav -> focusability score: 0.96 label: 1
data/processed/study/Stranded-chosic.com_.wav -> focusability score: 0.997 label: 1
data/processed/general/99 Tales - Baby Out Of Jail.wav -> focusability score: 0.002 label: 0
data/processed/general/Ultimate Fantastic - Choose Love.wav -> focusability score: 0.138 label: 0


In [ ]:
import pandas as pd

X_s_all = scaler.transform(X)
proba_all = logreg.predict_proba(X_s_all)[:, 1]

df_scores = pd.DataFrame({
    "path": paths,
    "label": y,
    "set": np.where(y == 1, "study", "general"),
    "focus_score": proba_all,
})

scores_path = BASE_DIR / "data" / "focus_scores_logreg.csv"
scores_path.parent.mkdir(parents=True, exist_ok=True)
df_scores.to_csv(scores_path, index=False)

print("[INFO] Saved scores to:", scores_path)
df_scores.head(10)


[INFO] Saved scores to: /content/_MusicProject/data/focus_scores_logreg.csv


,path,label,set,focus_score
0,data/processed/study/After_Hours-chosic.com_(c...,1,study,0.919855
1,data/processed/study/Afternoon-Nap-Lofi-Study-...,1,study,0.891543
2,data/processed/study/Arnor(chosic.com).wav,1,study,0.953690
3,data/processed/study/Balynt-Afternoon(chosic.c...,1,study,0.993357
4,data/processed/study/Beach-Club-chosic.com_.wav,1,study,0.998842
5,data/processed/study/Bedhead(chosic.com).wav,1,study,0.984870
6,data/processed/study/Bird-Bath(chosic.com).wav,1,study,0.997607
7,data/processed/study/Bubble-Wand-chosic.com_.wav,1,study,0.806017
8,data/processed/study/Cats-Cradle(chosic.com).wav,1,study,0.970450
9,data/processed/study/Celestial-Sands(chosic.co...,1,study,0.997822


In [ ]:
sample_file = next((FEATURE_DIR / "study").glob("*.npz"), None)
if sample_file is None:
    sample_file = next((FEATURE_DIR / "general").glob("*.npz"))

sample = np.load(sample_file, allow_pickle=True)
feature_names = list(sample["feature_names"])

coefs = logreg.coef_.ravel()
coef_df = pd.DataFrame({"feature": feature_names, "weight": coefs})
coef_df_sorted = coef_df.sort_values("weight", ascending=False)

top_positive = coef_df_sorted.head(10)
top_negative = coef_df_sorted.tail(10)

print("Top 10 features that INCREASE focus score (more study-like):")
display(top_positive)

print("\nTop 10 features that DECREASE focus score (more general-like):")
display(top_negative)


Top 10 features that INCREASE focus score (more study-like):


,feature,weight
34,mfcc_std_3,1.237150
13,rms_std,1.068664
20,mfcc_mean_2,0.782227
40,mfcc_std_9,0.764821
33,mfcc_std_2,0.692553
24,mfcc_mean_6,0.549279
42,mfcc_std_11,0.481996
1,tempo_bpm,0.438278
10,spec_flux_std,0.358142
3,tempo_stability_std,0.335847



Top 10 features that DECREASE focus score (more general-like):


,feature,weight
32,mfcc_std_1,-0.465891
26,mfcc_mean_8,-0.479761
11,spec_flatness_mean,-0.488910
6,spec_centroid_mean,-0.531923
7,spec_centroid_std,-0.587839
23,mfcc_mean_5,-0.641886
17,mfcc_temporal_var_std,-0.650112
31,mfcc_std_0,-0.750753
5,onset_density_var,-0.841447
37,mfcc_std_6,-0.895368


In [ ]:
from google.colab import files

scores_path = BASE_DIR / "data" / "focus_scores_logreg.csv"
scaler_path = BASE_DIR / "models" / "focus_scaler_logreg.joblib"
model_path = BASE_DIR / "models" / "focus_logreg.joblib"

print("Downloading:", scores_path)
files.download(str(scores_path))

print("Downloading:", scaler_path)
files.download(str(scaler_path))

print("Downloading:", model_path)
files.download(str(model_path))

Downloading: /content/_MusicProject/data/focus_scores_logreg.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: /content/_MusicProject/models/focus_scaler_logreg.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: /content/_MusicProject/models/focus_logreg.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>